In [4]:
import pandas as pd

path = '../data/raw/US_Regional_Sales_Data.csv'
df = pd.read_csv(path)

def section(title):
    print(f"\n{'=' * 60}\n{title}\n{'=' * 60}")

# 1. Shape
section("SHAPE")
print(f"Rows:    {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"Exact duplicate rows: {df.duplicated().sum():,}")

# 2. Column summary: dtype, nulls, unique count
section("COLUMN SUMMARY")
summary = pd.DataFrame({
    'Dtype':    df.dtypes.astype(str),
    'Nulls':    df.isna().sum(),
    'Null %':   (df.isna().mean() * 100).round(2),
    'Unique':   df.nunique(),
})
print(summary.to_string())

# 3. Sample rows
section("SAMPLE ROWS (first 3)")
print(df.head(3).T.to_string(header=['Row 0', 'Row 1', 'Row 2']))


SHAPE
Rows:    7,991
Columns: 16
Exact duplicate rows: 0

COLUMN SUMMARY
                    Dtype  Nulls  Null %  Unique
OrderNumber        object      0     0.0    7991
Sales Channel      object      0     0.0       4
WarehouseCode      object      0     0.0       6
ProcuredDate       object      0     0.0      11
OrderDate          object      0     0.0     945
ShipDate           object      0     0.0     966
DeliveryDate       object      0     0.0     966
CurrencyCode       object      0     0.0       1
_SalesTeamID        int64      0     0.0      28
_CustomerID         int64      0     0.0      50
_StoreID            int64      0     0.0     367
_ProductID          int64      0     0.0      47
Order Quantity      int64      0     0.0       8
Discount Applied  float64      0     0.0       7
Unit Cost          object      0     0.0    5252
Unit Price         object      0     0.0     664

SAMPLE ROWS (first 3)
                         Row 0         Row 1         Row 2
OrderNumber

In [3]:
d = df.copy()

date_cols = ['ProcuredDate', 'OrderDate', 'ShipDate', 'DeliveryDate']
money_cols = ['Unit Cost', 'Unit Price']

# Parse dates (day-first) and money (strip thousands separators)
for c in date_cols:
    d[c + '_p'] = pd.to_datetime(d[c], format='%d-%m-%Y', errors='coerce')
for c in money_cols:
    d[c + '_p'] = pd.to_numeric(d[c].astype(str).str.replace(',', '', regex=False), errors='coerce')

def section(title):
    print(f"\n{'=' * 50}\n{title}\n{'=' * 50}")

# 1. Overview
section("DATASET OVERVIEW")
print(f"Rows:                {len(df):,}")
print(f"Unique order numbers: {df['OrderNumber'].nunique():,}")
print(f"Duplicate order nums: {df['OrderNumber'].duplicated().sum():,}")
print(f"Warehouses:           {df['WarehouseCode'].nunique()}")

# 2. Date ranges
section("DATE RANGES & PARSE FAILURES")
dates = pd.DataFrame({
    c: {
        'Min': d[c + '_p'].min().date() if d[c + '_p'].notna().any() else None,
        'Max': d[c + '_p'].max().date() if d[c + '_p'].notna().any() else None,
        'Unparsed': int(d[c + '_p'].isna().sum()),
    } for c in date_cols
}).T
print(dates.to_string())

# 3. Date logic checks
section("DATE LOGIC CHECKS (should all be 0)")
checks = pd.Series({
    'Procured after order':   (d['ProcuredDate_p'] > d['OrderDate_p']).sum(),
    'Order after ship':       (d['OrderDate_p'] > d['ShipDate_p']).sum(),
    'Ship after delivery':    (d['ShipDate_p'] > d['DeliveryDate_p']).sum(),
    'Delivery before order':  (d['DeliveryDate_p'] < d['OrderDate_p']).sum(),
}, name='Violations').astype(int)
print(checks.to_string())

# 4. Money
section("MONEY COLUMNS")
money = pd.DataFrame({
    c: {
        'Min': d[c + '_p'].min(),
        'Max': d[c + '_p'].max(),
        'Unparsed': int(d[c + '_p'].isna().sum()),
    } for c in money_cols
}).T
print(money.round(2).to_string())

# 5. Quantity & discount
section("QUANTITY & DISCOUNT RANGE")
print(f"Order Quantity:   {df['Order Quantity'].min()} to {df['Order Quantity'].max()}")
print(f"Discount Applied: {df['Discount Applied'].min():.2f} to {df['Discount Applied'].max():.2f}")

# 6. Categorical counts
for col, title in [('CurrencyCode', 'CURRENCY'), ('Sales Channel', 'SALES CHANNELS')]:
    section(title)
    vc = df[col].value_counts()
    print(pd.DataFrame({'Count': vc, '%': (vc / vc.sum() * 100).round(1)}).to_string())


DATASET OVERVIEW
Rows:                7,991
Unique order numbers: 7,991
Duplicate order nums: 0
Warehouses:           6

DATE RANGES & PARSE FAILURES
                     Min         Max Unparsed
ProcuredDate  2017-12-31  2020-09-26        0
OrderDate     2018-05-31  2020-12-30        0
ShipDate      2018-06-02  2021-01-24        0
DeliveryDate  2018-06-07  2021-02-02        0

DATE LOGIC CHECKS (should all be 0)
Procured after order     0
Order after ship         0
Ship after delivery      0
Delivery before order    0

MONEY COLUMNS
               Min      Max  Unparsed
Unit Cost    68.68  5498.56       0.0
Unit Price  167.50  6566.00       0.0

QUANTITY & DISCOUNT RANGE
Order Quantity:   1 to 8
Discount Applied: 0.05 to 0.40

CURRENCY
              Count      %
CurrencyCode              
USD            7991  100.0

SALES CHANNELS
               Count     %
Sales Channel             
In-Store        3298  41.3
Online          2425  30.3
Distributor     1375  17.2
Wholesale        893